In [1]:
import pandas as pd
import os
from collections import defaultdict
from tqdm import tqdm
import requests
from datetime import datetime, timedelta
import time
import sys

In [20]:
df = pd.read_csv("processed_data/ncaa_basketball_processed_2003_2023.csv")

# Adding individual columns for attempts and makes
df[['FGM', 'FGA']] = df['field_goals_made_field_goals_attempted'].str.split('-', expand=True).astype(float)
df[['3PM', '3PA']] = df['three_point_field_goals_made_three_point_field_goals_attempted'].str.split('-', expand=True).astype(float)
df[['FTM', 'FTA']] = df['free_throws_made_free_throws_attempted'].str.split('-', expand=True).astype(float)

# Renaming columns to make things more concise
rename_map = {
    "game_id" : "Game_ID",
    "season" : "Season",
    "season_type" : "Season_Type",
    "game_date" : "Date",
    "game_date_time" : "Datetime",
    "team_location" : "Team",
    "opponent_team_location" : "Opponent",
    "home_away_combined" : "Site",
    "team_id" : "Team_ID",
    "opponent_id" : "Opp_ID",
    "team_score" : "Team_Score",
    "opponent_team_score" : "Opp_Score",
    "team_winner" : "Team_Win",
    "largest_lead" : "Largest_Lead",
    "field_goal_pct" : "FG%",
    "three_point_field_goal_pct" : "3P%",
    "free_throw_pct" : "FT%",
    "total_rebounds" : "TRB",
    "offensive_rebounds" : "ORB",
    "assists" : "AST",
    "steals" : "STL",
    "blocks" : "BLK",
    "total_turnovers" : "TOV",
    "fouls" : "PF"
}

df.rename(columns=rename_map, inplace=True)


# Columns we want opponent versions of
stat_cols = ["FGM", "FGA", "FG%", "3PM", "3PA", "3P%", "FTM", "FTA", "FT%", 
             "ORB", "TRB", "AST", "STL", "BLK", "TOV", "PF"]

# Perform self-merge on Game_ID
merged = df.merge(
    df[["Game_ID", "Team"] + stat_cols],
    on="Game_ID",
    suffixes=("", "_opp")
)

# Keep only rows where the opponent team is not the same team
merged = merged[merged["Team"] != merged["Team_opp"]]

# Rename opponent columns to start with 'o'
for col in stat_cols:
    merged.rename(columns={f"{col}_opp": f"o{col}"}, inplace=True)

# Drop the now-unneeded 'Team_opp' column
merged.drop(columns=["Team_opp","field_goals_made_field_goals_attempted","three_point_field_goals_made_three_point_field_goals_attempted","free_throws_made_free_throws_attempted","defensive_rebounds","team_rebounds"], inplace=True)

# Make sure the output folder exists
os.makedirs("processed_year_data", exist_ok=True)

# Split merged DataFrame by season and save each as a separate CSV
for season, group in merged.groupby("Season"):
    # Optional: reset index for each CSV
    group = group.reset_index(drop=True)
    
    # Create a filename per season
    filename = f"processed_year_data/cbb_{season}.csv"
    
    # Save to CSV
    group.to_csv(filename, index=False)
    
    print(f"Saved {filename} with {len(group)} rows.")


C:\Users\dschr\AppData\Local\Temp\ipykernel_22732\311535097.py:1: DtypeWarning: Columns (26) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("processed_data/ncaa_basketball_processed_2003_2023.csv")


Saved processed_year_data/cbb_2003.csv with 2 rows.
Saved processed_year_data/cbb_2004.csv with 24 rows.
Saved processed_year_data/cbb_2005.csv with 8836 rows.
Saved processed_year_data/cbb_2006.csv with 9846 rows.
Saved processed_year_data/cbb_2007.csv with 10488 rows.
Saved processed_year_data/cbb_2008.csv with 11094 rows.
Saved processed_year_data/cbb_2009.csv with 11278 rows.
Saved processed_year_data/cbb_2010.csv with 11496 rows.
Saved processed_year_data/cbb_2011.csv with 11294 rows.
Saved processed_year_data/cbb_2012.csv with 11288 rows.
Saved processed_year_data/cbb_2013.csv with 11368 rows.
Saved processed_year_data/cbb_2014.csv with 11640 rows.
Saved processed_year_data/cbb_2015.csv with 11628 rows.
Saved processed_year_data/cbb_2016.csv with 11646 rows.
Saved processed_year_data/cbb_2017.csv with 11610 rows.
Saved processed_year_data/cbb_2018.csv with 11798 rows.
Saved processed_year_data/cbb_2019.csv with 11876 rows.
Saved processed_year_data/cbb_2020.csv with 11512 rows.
S

In [56]:
valid_teams = set()
data_teams = set()

for year in range(2003,2024):
    filename = f"data/teams/{year}_teams.csv"
    data = pd.read_csv(filename)

    teams = list(data["Team"])

    for team in teams:
        valid_teams.add(team)

for year in range(2003,2024):
    filename = f"processed_year_data/cbb_{year}.csv"
    data = pd.read_csv(filename)

    teams = list(data["Team"].unique())

    for team in teams:
        data_teams.add(team)

print("VALID TEAMS")
print(sorted(list(valid_teams)))
print("DATA TEAMS")
print(sorted(list(data_teams)))

VALID TEAMS
['Abilene Christian', 'Air Force', 'Akron', 'Alabama', 'Alabama A&M', 'Alabama State', 'Albany (NY)', 'Alcorn State', 'American', 'Appalachian State', 'Arizona', 'Arizona State', 'Arkansas', 'Arkansas State', 'Arkansas-Pine Bluff', 'Army', 'Auburn', 'Austin Peay', 'Ball State', 'Baylor', 'Bellarmine', 'Belmont', 'Bethune-Cookman', 'Binghamton', 'Birmingham-Southern', 'Boise State', 'Boston College', 'Boston University', 'Bowling Green', 'Bradley', 'Brigham Young', 'Brown', 'Bryant', 'Bucknell', 'Buffalo', 'Butler', 'Cal Poly', 'Cal State Bakersfield', 'Cal State Fullerton', 'Cal State Northridge', 'California', 'California Baptist', 'Campbell', 'Canisius', 'Centenary (LA)', 'Central Arkansas', 'Central Connecticut State', 'Central Michigan', 'Charleston Southern', 'Charlotte', 'Chattanooga', 'Chicago State', 'Cincinnati', 'Clemson', 'Cleveland State', 'Coastal Carolina', 'Colgate', 'College of Charleston', 'Colorado', 'Colorado State', 'Columbia', 'Connecticut', 'Coppin Sta

In [60]:
"""
College Basketball Team Name Mapping
Maps DATA TEAMS names to VALID TEAMS names
"""
# Mapping dictionary from DATA TEAMS to VALID TEAMS
team_mapping = {
    # Exact matches (most common case)
    'Abilene Christian': 'Abilene Christian',
    'Air Force': 'Air Force',
    'Akron': 'Akron',
    'Alabama': 'Alabama',
    'Alabama A&M': 'Alabama A&M',
    'Alabama State': 'Alabama State',
    'Albany': 'Albany (NY)',
    'Alcorn State': 'Alcorn State',
    'American': 'American',
    'American University': 'American',
    'Appalachian State': 'Appalachian State',
    'Arizona': 'Arizona',
    'Arizona State': 'Arizona State',
    'Arkansas': 'Arkansas',
    'Arkansas State': 'Arkansas State',
    'Arkansas-Pine Bluff': 'Arkansas-Pine Bluff',
    'Army': 'Army',
    'Auburn': 'Auburn',
    'Austin Peay': 'Austin Peay',
    'Ball State': 'Ball State',
    'Baylor': 'Baylor',
    'Bellarmine': 'Bellarmine',
    'Belmont': 'Belmont',
    'Bethune-Cookman': 'Bethune-Cookman',
    'Binghamton': 'Binghamton',
    'Birmingham-Southern': 'Birmingham-Southern',
    'Boise State': 'Boise State',
    'Boston College': 'Boston College',
    'Boston University': 'Boston University',
    'Bowling Green': 'Bowling Green',
    'Bradley': 'Bradley',
    'BYU': 'Brigham Young',
    'Brown': 'Brown',
    'Bryant': 'Bryant',
    'Bucknell': 'Bucknell',
    'Buffalo': 'Buffalo',
    'Butler': 'Butler',
    'Cal Poly': 'Cal Poly',
    'Cal State Bakersfield': 'Cal State Bakersfield',
    'CSU Bakersfield': 'Cal State Bakersfield',
    'Cal State Fullerton': 'Cal State Fullerton',
    'CSU Fullerton': 'Cal State Fullerton',
    'Cal State Northridge': 'Cal State Northridge',
    'CSU Northridge': 'Cal State Northridge',
    'California': 'California',
    'California Baptist': 'California Baptist',
    'Campbell': 'Campbell',
    'Canisius': 'Canisius',
    'Centenary': 'Centenary (LA)',
    'Centenary Louisiana': 'Centenary (LA)',
    'Central Arkansas': 'Central Arkansas',
    'Central Connecticut': 'Central Connecticut State',
    'Central Michigan': 'Central Michigan',
    'Charleston': 'College of Charleston',
    'Charleston Southern': 'Charleston Southern',
    'Charlotte': 'Charlotte',
    'Chattanooga': 'Chattanooga',
    'Chicago State': 'Chicago State',
    'Cincinnati': 'Cincinnati',
    'Clemson': 'Clemson',
    'Cleveland State': 'Cleveland State',
    'Coastal Carolina': 'Coastal Carolina',
    'Colgate': 'Colgate',
    'Colorado': 'Colorado',
    'Colorado State': 'Colorado State',
    'Columbia': 'Columbia',
    'UConn': 'Connecticut',
    'Coppin State': 'Coppin State',
    'Cornell': 'Cornell',
    'Creighton': 'Creighton',
    'Dartmouth': 'Dartmouth',
    'Davidson': 'Davidson',
    'Dayton': 'Dayton',
    'DePaul': 'DePaul',
    'Delaware': 'Delaware',
    'Delaware State': 'Delaware State',
    'Denver': 'Denver',
    'Detroit Mercy': 'Detroit Mercy',
    'Drake': 'Drake',
    'Drexel': 'Drexel',
    'Duke': 'Duke',
    'Duquesne': 'Duquesne',
    'East Carolina': 'East Carolina',
    'East Tennessee State': 'East Tennessee State',
    'Eastern Illinois': 'Eastern Illinois',
    'Eastern Kentucky': 'Eastern Kentucky',
    'Eastern Michigan': 'Eastern Michigan',
    'Eastern Washington': 'Eastern Washington',
    'Elon': 'Elon',
    'Evansville': 'Evansville',
    'FDU-Florham': 'FDU',
    'Fairleigh Dickinson': 'FDU',
    'Fairfield': 'Fairfield',
    'Florida': 'Florida',
    'Florida A&M': 'Florida A&M',
    'Florida Atlantic': 'Florida Atlantic',
    'Florida Gulf Coast': 'Florida Gulf Coast',
    'Florida International': 'Florida International',
    'Florida State': 'Florida State',
    'Fordham': 'Fordham',
    'Fresno State': 'Fresno State',
    'Furman': 'Furman',
    'Gardner-Webb': 'Gardner-Webb',
    'George Mason': 'George Mason',
    'George Washington': 'George Washington',
    'Georgetown': 'Georgetown',
    'Georgia': 'Georgia',
    'Georgia Southern': 'Georgia Southern',
    'Georgia State': 'Georgia State',
    'Georgia Tech': 'Georgia Tech',
    'Gonzaga': 'Gonzaga',
    'Grambling': 'Grambling',
    'Grand Canyon': 'Grand Canyon',
    'Green Bay': 'Green Bay',
    'Hampton': 'Hampton',
    'Hartford': 'Hartford',
    'Harvard': 'Harvard',
    "Hawai'i": 'Hawaii',
    'High Point': 'High Point',
    'Hofstra': 'Hofstra',
    'Holy Cross': 'Holy Cross',
    'Houston': 'Houston',
    'Houston Baptist': 'Houston Christian',
    'Houston Christian': 'Houston Christian',
    'Howard': 'Howard',
    'IUPUI': 'IU Indy',
    'Idaho': 'Idaho',
    'Idaho State': 'Idaho State',
    'Illinois': 'Illinois',
    'Illinois State': 'Illinois State',
    'UIC': 'Illinois-Chicago',
    'Incarnate Word': 'Incarnate Word',
    'Indiana': 'Indiana',
    'Indiana State': 'Indiana State',
    'Iona': 'Iona',
    'Iowa': 'Iowa',
    'Iowa State': 'Iowa State',
    'Jackson State': 'Jackson State',
    'Jacksonville': 'Jacksonville',
    'Jacksonville State': 'Jacksonville State',
    'James Madison': 'James Madison',
    'Kansas': 'Kansas',
    'Kansas City': 'Kansas City',
    'UM Kansas City': 'Kansas City',
    'Kansas State': 'Kansas State',
    'Kennesaw State': 'Kennesaw State',
    'Kent State': 'Kent State',
    'Kentucky': 'Kentucky',
    'La Salle': 'La Salle',
    'Lafayette': 'Lafayette',
    'Lamar': 'Lamar',
    'Le Moyne': 'Le Moyne',
    'Lehigh': 'Lehigh',
    'Liberty': 'Liberty',
    'Lindenwood': 'Lindenwood',
    'Lipscomb': 'Lipscomb',
    'Little Rock': 'Little Rock',
    'Long Beach State': 'Long Beach State',
    'Long Island University': 'Long Island University',
    'Longwood': 'Longwood',
    'Louisiana': 'Louisiana',
    'LSU': 'Louisiana State',
    'Louisiana Tech': 'Louisiana Tech',
    'UL Monroe': 'Louisiana-Monroe',
    'Louisville': 'Louisville',
    'Loyola Chicago': 'Loyola (IL)',
    'Loyola (MD)': 'Loyola (MD)',
    'Loyola Maryland': 'Loyola (MD)',
    'Loyola Marymount': 'Loyola Marymount',
    'Maine': 'Maine',
    'Manhattan': 'Manhattan',
    'Marist': 'Marist',
    'Marquette': 'Marquette',
    'Marshall': 'Marshall',
    'Maryland': 'Maryland',
    'UMBC': 'Maryland-Baltimore County',
    'Maryland-Eastern Shore': 'Maryland-Eastern Shore',
    'UMass': 'Massachusetts',
    'UMass Lowell': 'Massachusetts-Lowell',
    'McNeese': 'McNeese State',
    'Memphis': 'Memphis',
    'Mercer': 'Mercer',
    'Merrimack': 'Merrimack',
    'Miami': 'Miami (FL)',
    'Miami (OH)': 'Miami (OH)',
    'Michigan': 'Michigan',
    'Michigan State': 'Michigan State',
    'Middle Tennessee': 'Middle Tennessee',
    'Milwaukee': 'Milwaukee',
    'Minnesota': 'Minnesota',
    'Ole Miss': 'Mississippi',
    'Mississippi State': 'Mississippi State',
    'Mississippi Valley State': 'Mississippi Valley State',
    'Missouri': 'Missouri',
    'Missouri State': 'Missouri State',
    'Monmouth': 'Monmouth',
    'Montana': 'Montana',
    'Montana State': 'Montana State',
    'Morehead State': 'Morehead State',
    'Morgan St': 'Morgan State',
    'Morgan State': 'Morgan State',
    "Mount St. Mary's": "Mount St. Mary's",
    "Mount St. Mary": "Mount St. Mary's",
    'Murray State': 'Murray State',
    'NC State': 'NC State',
    'NJIT': 'NJIT',
    'Navy': 'Navy',
    'Nebraska': 'Nebraska',
    'Nevada': 'Nevada',
    'UNLV': 'Nevada-Las Vegas',
    'New Hampshire': 'New Hampshire',
    'New Mexico': 'New Mexico',
    'New Mexico State': 'New Mexico State',
    'New Orleans': 'New Orleans',
    'Niagara': 'Niagara',
    'Nicholls': 'Nicholls State',
    'Norfolk State': 'Norfolk State',
    'North Alabama': 'North Alabama',
    'North Carolina': 'North Carolina',
    'North Carolina A&T': 'North Carolina A&T',
    'North Carolina Central': 'North Carolina Central',
    'North Dakota': 'North Dakota',
    'North Dakota State': 'North Dakota State',
    'North Florida': 'North Florida',
    'North Texas': 'North Texas',
    'Northeastern': 'Northeastern',
    'Northern Arizona': 'Northern Arizona',
    'Northern Colorado': 'Northern Colorado',
    'Northern Illinois': 'Northern Illinois',
    'Northern Iowa': 'Northern Iowa',
    'Northern Kentucky': 'Northern Kentucky',
    'Northwestern': 'Northwestern',
    'Northwestern State': 'Northwestern State',
    'Notre Dame': 'Notre Dame',
    'Oakland': 'Oakland',
    'Ohio': 'Ohio',
    'Ohio State': 'Ohio State',
    'Oklahoma': 'Oklahoma',
    'Oklahoma State': 'Oklahoma State',
    'Old Dominion': 'Old Dominion',
    'Omaha': 'Omaha',
    'Oral Roberts': 'Oral Roberts',
    'Oregon': 'Oregon',
    'Oregon State': 'Oregon State',
    'Pacific': 'Pacific',
    'Penn State': 'Penn State',
    'Pennsylvania': 'Pennsylvania',
    'Pepperdine': 'Pepperdine',
    'Pittsburgh': 'Pittsburgh',
    'Portland': 'Portland',
    'Portland State': 'Portland State',
    'Prairie View A&M': 'Prairie View',
    'Presbyterian': 'Presbyterian',
    'Princeton': 'Princeton',
    'Providence': 'Providence',
    'Purdue': 'Purdue',
    'Purdue Fort Wayne': 'Purdue Fort Wayne',
    'Queens NC': 'Queens (NC)',
    'Queens University': 'Queens (NC)',
    'Quinnipiac': 'Quinnipiac',
    'Radford': 'Radford',
    'Rhode Island': 'Rhode Island',
    'Rice': 'Rice',
    'Richmond': 'Richmond',
    'Rider': 'Rider',
    'Robert Morris': 'Robert Morris',
    'Rutgers': 'Rutgers',
    'SIU Edwardsville': 'SIU Edwardsville',
    'Sacramento State': 'Sacramento State',
    'Sacred Heart': 'Sacred Heart',
    'St. Francis (PA)': 'Saint Francis (PA)',
    "Saint Joseph's": "Saint Joseph's",
    'Saint Louis': 'Saint Louis',
    "Saint Mary's": "Saint Mary's (CA)",
    "Saint Peter's": "Saint Peter's",
    'Sam Houston': 'Sam Houston',
    'Sam Houston State': 'Sam Houston',
    'Samford': 'Samford',
    'San Diego': 'San Diego',
    'San Diego State': 'San Diego State',
    'San Francisco': 'San Francisco',
    'San José St': 'San Jose State',
    'San José State': 'San Jose State',
    'Santa Clara': 'Santa Clara',
    'Savannah State': 'Savannah State',
    'Seattle U': 'Seattle',
    'Seton Hall': 'Seton Hall',
    'Siena': 'Siena',
    'South Alabama': 'South Alabama',
    'South Carolina': 'South Carolina',
    'South Carolina State': 'South Carolina State',
    'South Carolina Upstate': 'South Carolina Upstate',
    'South Dakota': 'South Dakota',
    'South Dakota State': 'South Dakota State',
    'South Florida': 'South Florida',
    'Southeast Missouri State': 'Southeast Missouri State',
    'SE Louisiana': 'Southeastern Louisiana',
    'Southeastern': 'Southeastern Louisiana',
    'Southern': 'Southern',
    'USC': 'Southern California',
    'Southern Illinois': 'Southern Illinois',
    'Southern Indiana': 'Southern Indiana',
    'SMU': 'Southern Methodist',
    'Southern Miss': 'Southern Mississippi',
    'Southern Utah': 'Southern Utah',
    'St. Bonaventure': 'St. Bonaventure',
    'St. Francis Brooklyn': 'St. Francis (NY)',
    'St. Francis (BKN)': 'St. Francis (NY)',
    "St. John's": "St. John's (NY)",
    'St. Thomas - Minnesota': 'St. Thomas',
    'Stanford': 'Stanford',
    'Stephen F. Austin': 'Stephen F. Austin',
    'Stetson': 'Stetson',
    'Stonehill': 'Stonehill',
    'Stony Brook': 'Stony Brook',
    'Syracuse': 'Syracuse',
    'TCU': 'TCU',
    'Tarleton': 'Tarleton State',
    'Temple': 'Temple',
    'Tennessee': 'Tennessee',
    'Tennessee State': 'Tennessee State',
    'Tennessee Tech': 'Tennessee Tech',
    'UT Martin': 'Tennessee-Martin',
    'Texas': 'Texas',
    'Texas A&M': 'Texas A&M',
    'Texas A&M-CC': 'Texas A&M-Corpus Christi',
    'Texas A&M-Corpus Christi': 'Texas A&M-Corpus Christi',
    'Texas Southern': 'Texas Southern',
    'Texas State': 'Texas State',
    'Texas Tech': 'Texas Tech',
    'UT Rio Grande Valley': 'Texas-Rio Grande Valley',
    'The Citadel': 'The Citadel',
    'Toledo': 'Toledo',
    'Towson': 'Towson',
    'Troy': 'Troy',
    'Tulane': 'Tulane',
    'Tulsa': 'Tulsa',
    'UAB': 'UAB',
    'UC Davis': 'UC Davis',
    'UC Irvine': 'UC Irvine',
    'UC Riverside': 'UC Riverside',
    'UC San Diego': 'UC San Diego',
    'UC Santa Barbara': 'UC Santa Barbara',
    'UCF': 'UCF',
    'UCLA': 'UCLA',
    'UNC Asheville': 'UNC Asheville',
    'UNC Greensboro': 'UNC Greensboro',
    'UNC Wilmington': 'UNC Wilmington',
    'UT Arlington': 'UT Arlington',
    'UTEP': 'UTEP',
    'UTSA': 'UTSA',
    'Utah': 'Utah',
    'Utah State': 'Utah State',
    'Dixie State': 'Utah Tech',
    'Utah Tech': 'Utah Tech',
    'Utah Valley': 'Utah Valley',
    'VMI': 'VMI',
    'Valparaiso': 'Valparaiso',
    'Vanderbilt': 'Vanderbilt',
    'Vermont': 'Vermont',
    'Villanova': 'Villanova',
    'Virginia': 'Virginia',
    'VCU': 'Virginia Commonwealth',
    'Virginia Tech': 'Virginia Tech',
    'Wagner': 'Wagner',
    'Wake Forest': 'Wake Forest',
    'Washington': 'Washington',
    'Washington State': 'Washington State',
    'Weber State': 'Weber State',
    'West Virginia': 'West Virginia',
    'Western Carolina': 'Western Carolina',
    'Western Illinois': 'Western Illinois',
    'Western Kentucky': 'Western Kentucky',
    'Western Michigan': 'Western Michigan',
    'Wichita State': 'Wichita State',
    'William & Mary': 'William & Mary',
    'Winston Salem': 'Winston-Salem',
    'Winthrop': 'Winthrop',
    'Wisconsin': 'Wisconsin',
    'Wofford': 'Wofford',
    'Wright State': 'Wright State',
    'Wyoming': 'Wyoming',
    'Xavier': 'Xavier',
    'Yale': 'Yale',
    'Youngstown State': 'Youngstown State',
}

In [66]:
## Post processing the processed_year_data/ CSVs to sort for date and remove FCS games

for year in range(2003, 2024):
    filename = f"processed_year_data/cbb_{year}.csv"
    # print(f"Reading {filename}")
    data = pd.read_csv(filename)
    teams = pd.read_csv(f"data/teams/{year}_teams.csv")

    # Getting list of valid teams
    valid_teams = list(teams["Team"])
    
    # Map teams using team_mapping
    data["Team"] = data["Team"].map(team_mapping)
    data["Opponent"] = data["Opponent"].map(team_mapping)

    # Remove rows where mapping resulted in NaN (teams not in valid_teams)
    data = data.dropna(subset=["Team", "Opponent"])

    # Sorting by Dates
    data["Date"] = pd.to_datetime(data["Date"], errors="coerce")
    data = data.sort_values(by=["Team","Date"])

    # Save cleaned CSV
    data.to_csv(filename, index=False)

In [46]:
# Additional Processing
for year in range(2003, 2024):
    filename = f"processed_year_data/cbb_{year}.csv"

    df = pd.read_csv(filename)

    # Count occurrences of each Name
    name_counts = df['Team'].value_counts()

    # Keep only Names with at least 6 occurrences
    names_to_keep = name_counts[name_counts >= 6].index

    # Filter DataFrame
    df_filtered = df[df['Team'].isin(names_to_keep)]

    df_filtered.to_csv(filename, index=False)

In [49]:
# Define columns for cumulative stats
columns = ['Name', 'Game', 'Opponent', 'Site', 'Outcome', 'Date', 'ORtg', 'DRtg', '3PAr', 'TS%', 'TRB%', 'AST%', 'STL%', 'BLK%', 'eFG%', 
            'TOV%', 'ORB%', 'FTr', 'oeFG%', 'oTOV%', 'oDRB%', 'oFTr', 'Pace', 'o3PAr']

for i in range(2005, 2024):
    year = str(i)

    # Read data from CSV
    data = pd.read_csv("processed_year_data/cbb_" + year + ".csv")

    # Initialize/Reset dictionary to store totals data
    team_stats = defaultdict(lambda: {
        'Games': 0, 'Points': 0, 'oPoints': 0, 'FGA': 0, 'FGM': 0, '3PA': 0, '3PM': 0,
        'FTA': 0, 'FTM': 0, 'ORB': 0, 'TRB': 0, 'AST': 0, 'STL': 0, 'BLK': 0, 'TOV': 0,
        'Poss': 0, 'oFGA': 0, 'oFGM': 0, 'o3PA': 0, 'o3PM': 0, 'oFTA': 0, 'oFTM': 0,
        'oORB': 0, 'oTRB': 0, 'oTOV': 0, 'oPoss': 0
    })

    # Initialize a list to store all team rows for this year
    cumulative_stats_list = []

    # Loop through each row of data to aggregate stats by team
    for i in tqdm(range(len(data)), desc=year, unit="game"):
        team_name = data.iloc[i]['Team']
        
        # Convert each item in the current row to a float if it looks like a number
        data.iloc[i] = data.iloc[i].apply(lambda x: float(x) if isinstance(x, str) and x.replace('.', '', 1).isdigit() else x)

        # Count the number of successfully converted floats
        float_count = sum(isinstance(value, float) for value in data.iloc[i])

        # Count the number of empty or NaN values in the row
        empty_count = data.iloc[i].isnull().sum()

        # Skip the row if it has no convertible floats or if it has more than 1 empty value
        if float_count == 0 or empty_count > 2:
            continue

        # Update team data in the dictionary
        team_stats[team_name]['Games'] += 1
        team_stats[team_name]['Points'] += data.iloc[i]['Team_Score']
        team_stats[team_name]['FGA'] += data.iloc[i]['FGA']
        team_stats[team_name]['FGM'] += data.iloc[i]['FGM']
        team_stats[team_name]['3PA'] += data.iloc[i]['3PA']
        team_stats[team_name]['3PM'] += data.iloc[i]['3PM']
        team_stats[team_name]['FTA'] += data.iloc[i]['FTA']
        team_stats[team_name]['FTM'] += data.iloc[i]['FTM']
        team_stats[team_name]['ORB'] += data.iloc[i]['ORB']
        team_stats[team_name]['TRB'] += data.iloc[i]['TRB']
        team_stats[team_name]['AST'] += data.iloc[i]['AST']
        team_stats[team_name]['STL'] += data.iloc[i]['STL']
        team_stats[team_name]['BLK'] += data.iloc[i]['BLK']
        team_stats[team_name]['TOV'] += data.iloc[i]['TOV']
        team_stats[team_name]['Poss'] += data.iloc[i]['FGA'] - data.iloc[i]['ORB'] + data.iloc[i]['TOV'] + 0.475 * data.iloc[i]['FTA']
        team_stats[team_name]['oPoints'] += data.iloc[i]['Opp_Score']
        team_stats[team_name]['oFGA'] += data.iloc[i]['oFGA']
        team_stats[team_name]['oFGM'] += data.iloc[i]['oFGM']
        team_stats[team_name]['o3PA'] += data.iloc[i]['o3PA']
        team_stats[team_name]['o3PM'] += data.iloc[i]['o3PM']
        team_stats[team_name]['oFTA'] += data.iloc[i]['oFTA']
        team_stats[team_name]['oFTM'] += data.iloc[i]['oFTM']
        team_stats[team_name]['oORB'] += data.iloc[i]['oORB']
        team_stats[team_name]['oTRB'] += data.iloc[i]['oTRB']
        team_stats[team_name]['oTOV'] += data.iloc[i]['oTOV']
        team_stats[team_name]['oPoss'] += data.iloc[i]['oFGA'] - data.iloc[i]['oORB'] + data.iloc[i]['oTOV'] + 0.475 * data.iloc[i]['oFTA']

        # Convert only the current team's stats to a DataFrame
        current_team_stats = pd.DataFrame([{'Game ID' : data.iloc[i]['Game_ID'],
                                            'Name' : team_name,
                                            'Site' : data.iloc[i]['Site'],
                                            'Date' : data.iloc[i]['Date'],
                                            'Opponent' : data.iloc[i]['Opponent'],
                                            'Outcome' : int(data.iloc[i]['Team_Score'] > data.iloc[i]['Opp_Score']),
                                            'Game' : team_stats[team_name]['Games'],
                                            'ORtg' : 100 * (team_stats[team_name]['Points'] / team_stats[team_name]['Poss']),
                                            'DRtg' : 100 * (team_stats[team_name]['oPoints'] / team_stats[team_name]['oPoss']),
                                            '3PAr' : team_stats[team_name]['3PA'] / team_stats[team_name]['FGA'],
                                            'TS%' : team_stats[team_name]['Points'] / (2 * (team_stats[team_name]['FGA'] + 0.475 * team_stats[team_name]['FTA'])),
                                            'TRB%' : team_stats[team_name]['TRB'] / (team_stats[team_name]['TRB'] + team_stats[team_name]['oTRB']),
                                            'AST%' : team_stats[team_name]['AST'] / team_stats[team_name]['FGM'],
                                            'STL%' : team_stats[team_name]['STL'] / team_stats[team_name]['oPoss'],
                                            'BLK%' : team_stats[team_name]['BLK'] / (team_stats[team_name]['oFGA'] - team_stats[team_name]['o3PA']),
                                            'eFG%' : (team_stats[team_name]['FGM'] + 0.5 * team_stats[team_name]['3PM']) / team_stats[team_name]['FGA'],
                                            'TOV%' : team_stats[team_name]['TOV'] / team_stats[team_name]['Poss'],
                                            'ORB%' : team_stats[team_name]['ORB'] / (team_stats[team_name]['ORB'] + team_stats[team_name]['oTRB'] - team_stats[team_name]['oORB']),
                                            'FTr' : team_stats[team_name]['FTA'] / team_stats[team_name]['FGA'],
                                            'oeFG%' : (team_stats[team_name]['oFGM'] + 0.5 * team_stats[team_name]['o3PM']) / team_stats[team_name]['oFGA'],
                                            'oTOV%' : team_stats[team_name]['TOV'] / team_stats[team_name]['oPoss'],
                                            'oDRB%' : (team_stats[team_name]['oTRB'] - team_stats[team_name]['oORB']) / (team_stats[team_name]['oTRB'] - team_stats[team_name]['oORB'] + team_stats[team_name]['ORB']),
                                            'oFTr' : team_stats[team_name]['oFTA'] / team_stats[team_name]['oFGA'],
                                            'Pace' : (team_stats[team_name]['Poss'] + team_stats[team_name]['oPoss']),
                                            'o3PAr' : team_stats[team_name]['o3PA'] / team_stats[team_name]['oFGA']}])

        # Append the row to the list instead of concatenating
        cumulative_stats_list.append(current_team_stats)

    # Concatenate all rows at once after the loop
    cumulative_stats = pd.concat(cumulative_stats_list, ignore_index=True)

    # Sort by team name and then by games played within each team
    cumulative_stats.sort_values(by=['Name', 'Game'], inplace=True)

    cumulative_stats.to_csv('AdvWeek_CSVs/AdvWeekData' + year + '.csv', index=False, header=True)

2023: 100%|██████████| 11307/11307 [01:30<00:00, 124.37game/s]


In [51]:
# Combining with Sports Reference data to get MM games for 2011 - 2021

team_mapping = {
    'VCU' : 'Virginia Commonwealth',
    'UNLV' : 'Nevada-Las Vegas',
    'LIU Brooklyn' : 'Long Island University',
    'St. Mary\'s (CA)' : 'Saint Mary\'s (CA)',
    'Ole Miss' : 'Mississippi',
    'Louisiana–Lafayette' : 'Lafayette',
    'SMU' : 'Southern Methodist',
    'UMBC' : 'Maryland-Baltimore County',
    'Loyola Chicago' : 'Loyola (IL)',
    'Fairleigh Dickinson' : 'FDU',
}

for year in range(2011,2022):
    ref_data = pd.read_csv(f"extra_data/{year}GameData.csv")
    adv_data = pd.read_csv(f"AdvWeek_CSVs/AdvWeekData{year}.csv")

    # Filtering to only tournament games
    ref_data = ref_data[ref_data['Type'].str.contains('ROUND|NATIONAL', case=False, na=False)]

    ref_data['Team'] = ref_data['Team'].map(team_mapping).fillna(ref_data['Team'])
    ref_data['Opponent'] = ref_data['Opponent'].map(team_mapping).fillna(ref_data['Opponent'])

    new_rows = []

    for index, row in ref_data.iterrows():
        team = row['Team']

        # Getting team's data
        team_data = adv_data[adv_data['Name'] == team].copy()

        # Checking for empty rows
        if team_data.empty:
            print(f"Team: {team} {year}")
            sys.exit(1)

        # Ensure 'Game' is numeric
        team_data['Game'] = pd.to_numeric(team_data['Game'], errors='coerce')
        team_data = team_data.dropna(subset=['Game'])

        # Get index of the row with maximum 'Game'
        team_max_index = team_data['Game'].idxmax()

        # Get the row itself
        team_max_row = team_data.loc[team_max_index].copy()

        # Fix some columns
        team_max_row['Game ID'] = 0
        team_max_row['Game'] = team_max_row['Game'] + 1
        team_max_row['Opponent'] = row['Opponent']
        team_max_row['Site'] = 'neutral'
        team_max_row['Outcome'] = row['W/L']
        team_max_row['Date'] = row['Date']

        # Add to new rows array
        new_rows.append(team_max_row)

    # Concatenate all new rows at once
    if new_rows:
        new_rows_df = pd.DataFrame(new_rows)
        adv_data = pd.concat([adv_data, new_rows_df], ignore_index=True)

    # Sort by game and date
    adv_data = adv_data.sort_values(by=['Name','Date'])

    # Write to CSV
    adv_data.to_csv(f"AdvWeek_CSVs/AdvWeekData{year}.csv", index=False)

In [ ]:
# Post Processing Adv Week Data
# - Adding is_tournament_game variable

for year in range(2011, 2024):
    data = pd.read_csv(f"AdvWeek_CSVs/AdvWeekData{year}.csv")
    schedule = pd.read_csv(f"data/schedules/mbb_schedule_{year}.csv")

    is_tournament = schedule[schedule["tournament_id"] == 22]["id"]
    tournament_game_ids = set(is_tournament)
    tournament_game_ids.add(0)
    tournament_game_ids.discard(290762011) # Specifically removes 2009 pre-tournament game
    tournament_game_ids.discard(300752737) # Specifically removes 2010 pre-tournament game

    # Set is_tournament_game based on whether game_id is in the tournament set
    data["is_tournament_game"] = data["Game ID"].isin(tournament_game_ids)

    # Printing number of tournament games per year
    num_tournament_games = data["is_tournament_game"].sum()
    print(f"{year} {num_tournament_games}")

    # # Fixing Game IDs for 2011 - 2021
    # games_with_zero_id = data[data["Game ID"] == 0]
    # for idx, game in games_with_zero_id.iterrows():
    #     # print(game)
    #     schedule_game = schedule[
    #         (schedule["home_short_display_name"] == game["Name"]) &
    #         (schedule["away_short_display_name"] == game["Opponent"]) &
    #         (pd.to_datetime(schedule["date"]) == pd.to_datetime(game["Date"]))
    #     ]
    #     print(schedule_game)
    #     if not schedule_game.empty:
    #         data.at[idx, "Game ID"] = schedule_game.iloc[0]["id"]
    #     else:
    #         schedule_game2 = schedule[
    #             (schedule["home_short_display_name"] == game["Opponent"]) &
    #             (schedule["away_short_display_name"] == game["Name"]) &
    #             (pd.to_datetime(schedule["date"]) == pd.to_datetime(game["Date"]))
    #         ]
    #         if not schedule_game2.empty:
    #             data.at[idx, "Game ID"] = schedule_game2.iloc[0]["id"]

    # data.to_csv(f"{year}test.csv", index=False)

    # break
    
    data.to_csv(f"AdvWeek_CSVs/AdvWeekData{year}.csv", index=False)

## Missing Data
# - 2007: Virginia vs. Albany
# - 2007: Kansas vs. Niagara
# - 2020: No Tournament Happened (COVID)
# - 2021: Oregon vs. VCU (COVID disqualified VCU)
# - 2021: Drexel's stats for Illinois vs. Drexel

In [13]:
# Creating input data for model

# Define columns for input data
columns = ['Game ID', 'Team 1', 'Team 2', 'Date', 'Site', 'Outcome', '1-ORtg', '1-DRtg', '1-3PAr', '1-TS%', '1-TRB%',
           '1-AST%', '1-STL%', '1-BLK%', '1-eFG%', '1-TOV%', '1-ORB%', '1-FTr', '1-oeFG%', '1-oTOV%', '1-oDRB%',
           '1-oFTr', '1-Pace', '1-o3PAr', '2-ORtg', '2-DRtg', '2-3PAr', '2-TS%', '2-TRB%', '2-AST%', '2-STL%',
           '2-BLK%', '2-eFG%', '2-TOV%', '2-ORB%', '2-FTr', '2-oeFG%', '2-oTOV%', '2-oDRB%', '2-oFTr', '2-Pace', '2-o3PAr']

# Initialize a list to store all rows
input_data_list = []

for year in range(2023, 2024):
    advanced_stats = pd.read_csv('AdvWeek_CSVs/AdvWeekData' + str(year) + '.csv')

    yearly_data_list = []

    for j in tqdm(range(len(advanced_stats['Name'])), desc=str(year), unit="game"):
        team_data = advanced_stats.iloc[j]

        name = team_data['Name']
        opp = team_data['Opponent']
        date = team_data['Date']

        # Finding opponent's data
        opp_data = advanced_stats[(advanced_stats['Name'] == opp) & (advanced_stats['Date'] == date)]

        # Continue if opponent's data does not exist (Usually a non FBS School)
        if opp_data.empty:
            continue
        else:
            opp_data = opp_data.iloc[0]

        # Getting previous game data
        prev_team_data = advanced_stats[(advanced_stats['Name'] == name) & (advanced_stats['Game'] == int(team_data['Game']) - 1)]
        prev_opp_data = advanced_stats[(advanced_stats['Name'] == opp) & (advanced_stats['Game'] == int(opp_data['Game']) - 1)]

        if prev_team_data.empty or prev_opp_data.empty:
            continue

        prev_opp_data = prev_opp_data.iloc[0]
        prev_team_data = prev_team_data.iloc[0]

        current_stats = pd.DataFrame([{
            'Game ID': team_data['Game ID'],
            'Team 1': team_data['Name'],
            'Team 2': team_data['Opponent'],
            'Date': team_data['Date'],
            'Site': team_data['Site'],
            'Outcome': 1 if team_data['Outcome'] == 'W' else (0 if team_data['Outcome'] == 'L' else team_data['Outcome']),
            'is_tournament_game' : team_data['is_tournament_game'],
            '1-ORtg': prev_team_data['ORtg'],
            '1-DRtg': prev_team_data['DRtg'],
            '1-3PAr': prev_team_data['3PAr'],
            '1-TS%': prev_team_data['TS%'],
            '1-TRB%': prev_team_data['TRB%'],
            '1-AST%': prev_team_data['AST%'],
            '1-STL%': prev_team_data['STL%'],
            '1-BLK%': prev_team_data['BLK%'],
            '1-eFG%': prev_team_data['eFG%'],
            '1-TOV%': prev_team_data['TOV%'],
            '1-ORB%': prev_team_data['ORB%'],
            '1-FTr': prev_team_data['FTr'],
            '1-oeFG%': prev_team_data['oeFG%'],
            '1-oTOV%': prev_team_data['oTOV%'],
            '1-oDRB%': prev_team_data['oDRB%'],
            '1-oFTr': prev_team_data['oFTr'],
            '1-Pace': prev_team_data['Pace'],
            '1-o3PAr': prev_team_data['o3PAr'],
            '2-ORtg': prev_opp_data['ORtg'],
            '2-DRtg': prev_opp_data['DRtg'],
            '2-3PAr': prev_opp_data['3PAr'],
            '2-TS%': prev_opp_data['TS%'],
            '2-TRB%': prev_opp_data['TRB%'],
            '2-AST%': prev_opp_data['AST%'],
            '2-STL%': prev_opp_data['STL%'],
            '2-BLK%': prev_opp_data['BLK%'],
            '2-eFG%': prev_opp_data['eFG%'],
            '2-TOV%': prev_opp_data['TOV%'],
            '2-ORB%': prev_opp_data['ORB%'],
            '2-FTr': prev_opp_data['FTr'],
            '2-oeFG%': prev_opp_data['oeFG%'],
            '2-oTOV%': prev_opp_data['oTOV%'],
            '2-oDRB%': prev_opp_data['oDRB%'],
            '2-oFTr': prev_opp_data['oFTr'],
            '2-Pace': prev_opp_data['Pace'],
            '2-o3PAr': prev_opp_data['o3PAr']
        }])

        yearly_data_list.append(current_stats)

    # Concatenate yearly data
    yearly_data = pd.concat(yearly_data_list, ignore_index=True)

    # Remove mirrored duplicates (alphabetical order)
    yearly_data['Team_A'] = yearly_data[['Team 1', 'Team 2']].min(axis=1)
    yearly_data['Team_B'] = yearly_data[['Team 1', 'Team 2']].max(axis=1)
    yearly_data.sort_values(by=['Team_A', 'Team_B', 'Date'], inplace=True)
    yearly_data = yearly_data.drop_duplicates(subset=['Team_A', 'Team_B', 'Date'], keep='first')
    yearly_data.drop(columns=['Team_A', 'Team_B'], inplace=True)

    # Append to master list
    input_data_list.append(yearly_data)

# Concatenate all rows at once
input_data = pd.concat(input_data_list, ignore_index=True)

# Sort games by Date
input_data.sort_values(by=['Date'], inplace=True)

# Export Data to a csv
input_data.to_csv('InputData2023.csv', index=False, header=True)

2023: 100%|██████████| 11307/11307 [00:50<00:00, 222.24game/s]


In [36]:
# === CONFIGURATION ===
API_KEY = os.getenv("ODDS_API_KEY")
if API_KEY is None:
    raise ValueError("API key not found. Please set the ODDS_API_KEY environment variable.")

SPORT = "basketball_ncaab"
BOOKMAKERS = "fanduel,draftkings"
MARKETS = "h2h"
START_DATE = datetime(2020, 6, 6)
END_DATE = datetime.now()
OUTPUT_FILE = "historical_cbb_odds.csv"

# === SEASON LOGIC ===
def is_basketball_season(date):
    """
    Returns True if the date is roughly within the NCAA basketball season window.
    Adds a little buffer room before/after (Aug–May).
    """
    return date.month >= 8 or date.month <= 5  # Aug–May (buffered)

# === MAIN SCRIPT ===
all_odds = []
current_date = START_DATE

while current_date <= END_DATE:
    if not is_basketball_season(current_date):
        current_date += timedelta(days=1)
        continue

    date_str = current_date.strftime("%Y-%m-%dT12:00:00Z")
    print(f"Fetching odds for {date_str}...")

    url = f"https://api.the-odds-api.com/v4/historical/sports/{SPORT}/odds"
    params = {
        "apiKey": API_KEY,
        "bookmakers": BOOKMAKERS,
        "markets": MARKETS,
        "date": date_str,
    }

    try:
        response = requests.get(url, params=params)
    except requests.RequestException as e:
        print(f"⚠️ Request failed for {date_str}: {e}")
        time.sleep(1)
        current_date += timedelta(days=1)
        continue

    if response.status_code != 200:
        print(f"⚠️ Error {response.status_code} for {date_str}: {response.text[:200]}")
        current_date += timedelta(days=1)
        time.sleep(1)
        continue

    try:
        json_data = response.json()
    except ValueError:
        print(f"⚠️ Non-JSON response for {date_str}: {response.text[:200]}")
        current_date += timedelta(days=1)
        time.sleep(1)
        continue

    games = json_data.get("data", [])
    if not games:
        print(f"No games on {date_str}")
        current_date += timedelta(days=1)
        time.sleep(1)
        continue

    for game in games:
        for bookmaker in game.get("bookmakers", []):
            book_name = bookmaker.get("title")
            for market in bookmaker.get("markets", []):
                market_key = market.get("key")
                for outcome in market.get("outcomes", []):
                    all_odds.append({
                        "snapshot_date": date_str,
                        "game_id": game.get("id"),
                        "sport": game.get("sport_title"),
                        "home_team": game.get("home_team"),
                        "away_team": game.get("away_team"),
                        "bookmaker": book_name,
                        "market": market_key,
                        "team": outcome.get("name"),
                        "price": outcome.get("price"),
                        "point": outcome.get("point"),
                        "commence_time": game.get("commence_time"),
                        "last_update": bookmaker.get("last_update")
                    })
    print(f"✓ Added odds for {len(games)} games on {date_str}")

    current_date += timedelta(days=1)
    time.sleep(1)  # Avoid rate limit

# === SAVE RESULTS ===
if all_odds:
    df = pd.DataFrame(all_odds)
    df.to_csv(OUTPUT_FILE, index=False)
    print(f"\n✅ Saved {len(df)} rows to {OUTPUT_FILE}")
else:
    print("No odds data collected.")


Fetching odds for 2020-08-01T12:00:00Z...
No games on 2020-08-01T12:00:00Z
Fetching odds for 2020-08-02T12:00:00Z...
No games on 2020-08-02T12:00:00Z
Fetching odds for 2020-08-03T12:00:00Z...
No games on 2020-08-03T12:00:00Z
Fetching odds for 2020-08-04T12:00:00Z...
No games on 2020-08-04T12:00:00Z
Fetching odds for 2020-08-05T12:00:00Z...
No games on 2020-08-05T12:00:00Z
Fetching odds for 2020-08-06T12:00:00Z...
No games on 2020-08-06T12:00:00Z
Fetching odds for 2020-08-07T12:00:00Z...
No games on 2020-08-07T12:00:00Z
Fetching odds for 2020-08-08T12:00:00Z...
No games on 2020-08-08T12:00:00Z
Fetching odds for 2020-08-09T12:00:00Z...
No games on 2020-08-09T12:00:00Z
Fetching odds for 2020-08-10T12:00:00Z...
No games on 2020-08-10T12:00:00Z
Fetching odds for 2020-08-11T12:00:00Z...
No games on 2020-08-11T12:00:00Z
Fetching odds for 2020-08-12T12:00:00Z...
No games on 2020-08-12T12:00:00Z
Fetching odds for 2020-08-13T12:00:00Z...
No games on 2020-08-13T12:00:00Z
Fetching odds for 2020-08

In [13]:
odds_stats_mapping = {
    "Penn State Nittany Lions": "Penn State",
    "Boston Univ. Terriers": "Boston University",
    "Norfolk St Spartans": "Norfolk State",
    "Bowling Green Falcons": "Bowling Green",
    "Cal Baptist Lancers": "California Baptist",
    "Dayton Flyers": "Dayton",
    "Queens University Royals": "Queens (NC)",
    "NJIT Highlanders": "NJIT",
    "Mississippi St Bulldogs": "Mississippi State",
    "Portland St Vikings": "Portland State",
    "Brown Bears": "Brown",
    "San José St Spartans": "San Jose State",
    "Northern Illinois Huskies": "Northern Illinois",
    "Lafayette Leopards": "Lafayette",
    "Belmont Bruins": "Belmont",
    "Navy Midshipmen": "Navy",
    "South Dakota St Jackrabbits": "South Dakota State",
    "Seattle Redhawks": "Seattle",
    "South Alabama Jaguars": "South Alabama",
    "Arkansas St Red Wolves": "Arkansas State",
    "Northern Arizona Lumberjacks": "Northern Arizona",
    "Vermont Catamounts": "Vermont",
    "Michigan St Spartans": "Michigan State",
    "Western Michigan Broncos": "Western Michigan",
    "San Diego St Aztecs": "San Diego State",
    "SE Missouri St Redhawks": "Southeast Missouri State",
    "LSU Tigers": "LSU",
    "UL Monroe Warhawks": "Louisiana-Monroe",
    "Auburn Tigers": "Auburn",
    "SIU-Edwardsville Cougars": "SIU Edwardsville",
    "Omaha Mavericks": "Omaha",
    "Radford Highlanders": "Radford",
    "Western Illinois Leathernecks": "Western Illinois",
    "Cal Poly Mustangs": "Cal Poly",
    "UConn Huskies": "Connecticut",
    "Duquesne Dukes": "Duquesne",
    "Coppin St Eagles": "Coppin State",
    "Furman Paladins": "Furman",
    "Indiana Hoosiers": "Indiana",
    "Dixie State Trailblazers": "Utah Tech",
    "Washington St Cougars": "Washington State",
    "CSU Northridge Matadors": "Cal State Northridge",
    "Fairfield Stags": "Fairfield",
    "Tarleton State Texans": "Tarleton State",
    "UC San Diego Tritons": "UC San Diego",
    "North Carolina Central Eagles": "North Carolina Central",
    "Louisiana Tech Bulldogs": "Louisiana Tech",
    "Creighton Bluejays": "Creighton",
    "Minnesota Golden Gophers": "Minnesota",
    "Jacksonville St Gamecocks": "Jacksonville State",
    "Murray St Racers": "Murray State",
    "St. Bonaventure Bonnies": "St. Bonaventure",
    "Southern Illinois Salukis": "Southern Illinois",
    "Akron Zips": "Akron",
    "Pennsylvania Quakers": "Pennsylvania",
    "Old Dominion Monarchs": "Old Dominion",
    "Vanderbilt Commodores": "Vanderbilt",
    "Utah Tech Trailblazers": "Utah Tech",
    "Abilene Christian Wildcats": "Abilene Christian",
    "Houston Cougars": "Houston",
    "Albany Great Danes": "Albany (NY)",
    "Idaho Vandals": "Idaho",
    "Toledo Rockets": "Toledo",
    "TCU Horned Frogs": "TCU",
    "UCF Knights": "UCF",
    "Mercyhurst Lakers": "Mercyhurst",
    "Florida Atlantic Owls": "Florida Atlantic",
    "Fresno St Bulldogs": "Fresno State",
    "Dartmouth Big Green": "Dartmouth",
    "Central Arkansas Bears": "Central Arkansas",
    "Denver Pioneers": "Denver",
    "UNC Asheville Bulldogs": "UNC Asheville",
    "Florida A&M Rattlers": "Florida A&M",
    "UC Davis Aggies": "UC Davis",
    "BYU Cougars": "BYU",
    "Pittsburgh Panthers": "Pittsburgh",
    "Marquette Golden Eagles": "Marquette",
    "Mcneese St Mcneese": "McNeese State",
    "Jacksonville Dolphins": "Jacksonville",
    "Memphis Tigers": "Memphis",
    "Western Carolina Catamounts": "Western Carolina",
    "Kent State Golden Flashes": "Kent State",
    "Eastern Illinois Panthers": "Eastern Illinois",
    "Oregon St Beavers": "Oregon State",
    "Richmond Spiders": "Richmond",
    "Central Michigan Chippewas": "Central Michigan",
    "Winthrop Eagles": "Winthrop",
    "Youngstown St Penguins": "Youngstown State",
    "Niagara Purple Eagles": "Niagara",
    "Drake Bulldogs": "Drake",
    "Texas Longhorns": "Texas",
    "IUPUI Jaguars": "IU Indy",
    "Southern Utah Thunderbirds": "Southern Utah",
    "Wisconsin Badgers": "Wisconsin",
    "Florida Gators": "Florida",
    "George Mason Patriots": "George Mason",
    "New Orleans Privateers": "New Orleans",
    "Charlotte 49ers": "Charlotte",
    "Ohio Bobcats": "Ohio",
    "Xavier Musketeers": "Xavier",
    "Bradley Braves": "Bradley",
    "California Golden Bears": "California",
    "Fort Wayne Mastodons": "Purdue Fort Wayne",
    "Liberty Flames": "Liberty",
    "CSU Fullerton Titans": "Cal State Fullerton",
    "UT-Arlington Mavericks": "UT Arlington",
    "UNC Greensboro Spartans": "UNC Greensboro",
    "Massachusetts Minutemen": "Massachusetts",
    "Illinois St Redbirds": "Illinois State",
    "Purdue Boilermakers": "Purdue",
    "Manhattan Jaspers": "Manhattan",
    "UTEP Miners": "UTEP",
    "GW Revolutionaries": "George Washington",
    "Ole Miss Rebels": "Mississippi",
    "Montana Grizzlies": "Montana",
    "Merrimack Warriors": "Merrimack",
    "Georgia Southern Eagles": "Georgia Southern",
    "Jackson St Tigers": "Jackson State",
    "Texas A&M-CC Islanders": "Texas A&M-Corpus Christi",
    "Virginia Tech Hokies": "Virginia Tech",
    "Wichita St Shockers": "Wichita State",
    "Tennessee Tech Golden Eagles": "Tennessee Tech",
    "Arizona St Sun Devils": "Arizona State",
    "Stonehill Skyhawks": "Stonehill",
    "USC Trojans": "Southern California",
    "UIC Flames": "Illinois-Chicago",
    "Stony Brook Seawolves": "Stony Brook",
    "Maine Black Bears": "Maine",
    "South Dakota Coyotes": "South Dakota",
    "Campbell Fighting Camels": "Campbell",
    "Portland Pilots": "Portland",
    "Canisius Golden Griffins": "Canisius",
    "Baylor Bears": "Baylor",
    "UTSA Roadrunners": "UTSA",
    "Mt. St. Mary's Mountaineers": "Mount St. Mary's",
    "Temple Owls": "Temple",
    "Iowa State Cyclones": "Iowa State",
    "Oklahoma St Cowboys": "Oklahoma State",
    "Cincinnati Bearcats": "Cincinnati",
    "CSU Bakersfield Roadrunners": "Cal State Bakersfield",
    "Arkansas-Little Rock Trojans": "Little Rock",
    "Washington Huskies": "Washington",
    "Southern Indiana Screaming Eagles": "Southern Indiana",
    "Delaware St Hornets": "Delaware State",
    "New Mexico Lobos": "New Mexico",
    "Wofford Terriers": "Wofford",
    "Arkansas Razorbacks": "Arkansas",
    "Wright St Raiders": "Wright State",
    "Virginia Cavaliers": "Virginia",
    "UMBC Retrievers": "UMBC",
    "Wagner Seahawks": "Wagner",
    "North Texas Mean Green": "North Texas",
    "East Carolina Pirates": "East Carolina",
    "St. Thomas (MN) Tommies": "St. Thomas",
    "Eastern Kentucky Colonels": "Eastern Kentucky",
    "Davidson Wildcats": "Davidson",
    "West Virginia Mountaineers": "West Virginia",
    "Hartford Hawks": "Hartford",
    "Robert Morris Colonials": "Robert Morris",
    "Southern Jaguars": "Southern",
    "Saint Louis Billikens": "Saint Louis",
    "Miami (OH) RedHawks": "Miami (OH)",
    "South Carolina St Bulldogs": "South Carolina State",
    "Wake Forest Demon Deacons": "Wake Forest",
    "N Colorado Bears": "Northern Colorado",
    "Appalachian St Mountaineers": "Appalachian State",
    "Oral Roberts Golden Eagles": "Oral Roberts",
    "Northwestern St Demons": "Northwestern State",
    "Buffalo Bulls": "Buffalo",
    "Pacific Tigers": "Pacific",
    "Holy Cross Crusaders": "Holy Cross",
    "Eastern Washington Eagles": "Eastern Washington",
    "Charleston Cougars": "College of Charleston",
    "NC State Wolfpack": "NC State",
    "Miss Valley St Delta Devils": "Mississippi Valley State",
    "Austin Peay Governors": "Austin Peay",
    "Texas A&M Aggies": "Texas A&M",
    "Bethune-Cookman Wildcats": "Bethune-Cookman",
    "Gardner-Webb Bulldogs": "Gardner-Webb",
    "UC Santa Barbara Gauchos": "UC Santa Barbara",
    "Yale Bulldogs": "Yale",
    "Middle Tennessee Blue Raiders": "Middle Tennessee",
    "Hampton Pirates": "Hampton",
    "Presbyterian Blue Hose": "Presbyterian",
    "Marshall Thundering Herd": "Marshall",
    "West Georgia Wolves": "West Georgia",
    "DePaul Blue Demons": "DePaul",
    "Evansville Purple Aces": "Evansville",
    "Alabama St Hornets": "Alabama State",
    "Indiana St Sycamores": "Indiana State",
    "Drexel Dragons": "Drexel",
    "Alcorn St Braves": "Alcorn State",
    "Santa Clara Broncos": "Santa Clara",
    "St. John's Red Storm": "St. John's (NY)",
    "Villanova Wildcats": "Villanova",
    "Texas Tech Red Raiders": "Texas Tech",
    "Eastern Michigan Eagles": "Eastern Michigan",
    "Rhode Island Rams": "Rhode Island",
    "Valparaiso Crusaders": "Valparaiso",
    "San Diego Toreros": "San Diego",
    "Arkansas-Pine Bluff Golden Lions": "Arkansas-Pine Bluff",
    "New Hampshire Wildcats": "New Hampshire",
    "Alabama Crimson Tide": "Alabama",
    "Prairie View Panthers": "Prairie View",
    "SMU Mustangs": "SMU",
    "Tennessee St Tigers": "Tennessee State",
    "Alabama A&M Bulldogs": "Alabama A&M",
    "Georgetown Hoyas": "Georgetown",
    "Samford Bulldogs": "Samford",
    "Boise State Broncos": "Boise State",
    "St. Francis (PA) Red Flash": "Saint Francis (PA)",
    "James Madison Dukes": "James Madison",
    "Loyola (Chi) Ramblers": "Loyola (IL)",
    "SE Louisiana Lions": "Southeastern Louisiana",
    "George Washington Colonials": "George Washington",
    "LIU Sharks": "Long Island University",
    "Saint Peter's Peacocks": "Saint Peter's",
    "Seton Hall Pirates": "Seton Hall",
    "Elon Phoenix": "Elon",
    "Nicholls St Colonels": "Nicholls State",
    "Oregon Ducks": "Oregon",
    "Cornell Big Red": "Cornell",
    "Maryland Terrapins": "Maryland",
    "Montana St Bobcats": "Montana State",
    "Grand Canyon Antelopes": "Grand Canyon",
    "Sacred Heart Pioneers": "Sacred Heart",
    "Nebraska Cornhuskers": "Nebraska",
    "Saint Joseph's Hawks": "Saint Joseph's",
    "Georgia Bulldogs": "Georgia",
    "Ball State Cardinals": "Ball State",
    "Butler Bulldogs": "Butler",
    "Northwestern Wildcats": "Northwestern",
    "Clemson Tigers": "Clemson",
    "Southern Miss Golden Eagles": "Southern Mississippi",
    "Valparaiso Beacons": "Valparaiso",
    "UCLA Bruins": "UCLA",
    "Ohio State Buckeyes": "Ohio State",
    "Fordham Rams": "Fordham",
    "San Francisco Dons": "San Francisco",
    "Longwood Lancers": "Longwood",
    "Tennessee Volunteers": "Tennessee",
    "South Carolina Gamecocks": "South Carolina",
    "Princeton Tigers": "Princeton",
    "North Carolina A&T Aggies": "North Carolina A&T",
    "Iona Gaels": "Iona",
    "Syracuse Orange": "Syracuse",
    "Georgia St Panthers": "Georgia State",
    "Detroit Mercy Titans": "Detroit",
    "Rutgers Scarlet Knights": "Rutgers",
    "Utah State Aggies": "Utah State",
    "Georgia Tech Yellow Jackets": "Georgia Tech",
    "North Carolina Tar Heels": "North Carolina",
    "VCU Rams": "Virginia Commonwealth",
    "Weber State Wildcats": "Weber State",
    "Missouri Tigers": "Missouri",
    "Colorado St Rams": "Colorado State",
    "Illinois Fighting Illini": "Illinois",
    "UMass Lowell River Hawks": "Massachusetts-Lowell",
    "Charleston Southern Buccaneers": "Charleston Southern",
    "Hofstra Pride": "Hofstra",
    "Miami Hurricanes": "Miami (FL)",
    "Fairleigh Dickinson Knights": "Fairleigh Dickinson",
    "Chicago St Cougars": "Chicago State",
    "La Salle Explorers": "La Salle",
    "Green Bay Phoenix": "Green Bay",
    "Lipscomb Bisons": "Lipscomb",
    "Milwaukee Panthers": "Milwaukee",
    "Kennesaw St Owls": "Kennesaw State",
    "Western Kentucky Hilltoppers": "Western Kentucky",
    "Siena Saints": "Siena",
    "Texas A&M-Commerce Lions": "Texas A&M-Commerce",
    "East Tennessee St Buccaneers": "East Tennessee State",
    "Howard Bison": "Howard",
    "Stanford Cardinal": "Stanford",
    "Northern Iowa Panthers": "Northern Iowa",
    "Towson Tigers": "Towson",
    "Rider Broncs": "Rider",
    "Notre Dame Fighting Irish": "Notre Dame",
    "North Dakota St Bison": "North Dakota State",
    "Bellarmine Knights": "Bellarmine",
    "Quinnipiac Bobcats": "Quinnipiac",
    "Cleveland St Vikings": "Cleveland State",
    "LIU Brooklyn Blackbirds": "Long Island University",
    "Air Force Falcons": "Air Force",
    "William & Mary Tribe": "William & Mary",
    "New Mexico St Aggies": "New Mexico State",
    "Iowa Hawkeyes": "Iowa",
    "Morgan St Bears": "Morgan State",
    "Kansas St Wildcats": "Kansas State",
    "Le Moyne Dolphins": "Le Moyne",
    "Texas State Bobcats": "Texas State",
    "Chattanooga Mocs": "Chattanooga",
    "Michigan Wolverines": "Michigan",
    "North Alabama Lions": "North Alabama",
    "Boston College Eagles": "Boston College",
    "Pepperdine Waves": "Pepperdine",
    "Central Connecticut St Blue Devils": "Central Connecticut State",
    "Monmouth Hawks": "Monmouth",
    "Florida Int'l Golden Panthers": "Florida International",
    "Stephen F. Austin Lumberjacks": "Stephen F. Austin",
    "South Florida Bulls": "South Florida",
    "Idaho State Bengals": "Idaho State",
    "Utah Utes": "Utah",
    "Duke Blue Devils": "Duke",
    "Nevada Wolf Pack": "Nevada",
    "Oklahoma Sooners": "Oklahoma",
    "UT Rio Grande Valley Vaqueros": "Texas-Rio Grande Valley",
    "Houston Baptist Huskies": "Houston Christian",
    "Utah Valley Wolverines": "Utah Valley",
    "Kansas Jayhawks": "Kansas",
    "Colgate Raiders": "Colgate",
    "Stetson Hatters": "Stetson",
    "Loyola Marymount Lions": "Loyola Marymount",
    "High Point Panthers": "High Point",
    "Louisiana Ragin' Cajuns": "Louisiana",
    "Florida St Seminoles": "Florida State",
    "Marist Red Foxes": "Marist",
    "Grambling St Tigers": "Grambling",
    "Colorado Buffaloes": "Colorado",
    "Gonzaga Bulldogs": "Gonzaga",
    "Bryant Bulldogs": "Bryant",
    "The Citadel Bulldogs": "The Citadel",
    "Missouri St Bears": "Missouri State",
    "Loyola (MD) Greyhounds": "Loyola (MD)",
    "North Florida Ospreys": "North Florida",
    "Coastal Carolina Chanticleers": "Coastal Carolina",
    "Columbia Lions": "Columbia",
    "Sam Houston St Bearkats": "Sam Houston",
    "Troy Trojans": "Troy",
    "Morehead St Eagles": "Morehead State",
    "Lehigh Mountain Hawks": "Lehigh",
    "Delaware Blue Hens": "Delaware",
    "Long Beach St 49ers": "Long Beach State",
    "Saint Mary's Gaels": "Saint Mary's (CA)",
    "UNC Wilmington Seahawks": "UNC Wilmington",
    "Northeastern Huskies": "Northeastern",
    "UNLV Rebels": "Nevada-Las Vegas",
    "Bucknell Bison": "Bucknell",
    "Houston Christian Huskies": "Houston Christian",
    "Maryland-Eastern Shore Hawks": "Maryland-Eastern Shore",
    "Arizona Wildcats": "Arizona",
    "UC Riverside Highlanders": "UC Riverside",
    "Oakland Golden Grizzlies": "Oakland",
    "Rice Owls": "Rice",
    "Tenn-Martin Skyhawks": "UT Martin",
    "Wyoming Cowboys": "Wyoming",
    "Florida Gulf Coast Eagles": "Florida Gulf Coast",
    "South Carolina Upstate Spartans": "South Carolina Upstate",
    "UMKC Kangaroos": "Kansas City",
    "Hawai'i Rainbow Warriors": "Hawai'i",
    "Tulsa Golden Hurricane": "Tulsa",
    "Providence Friars": "Providence",
    "St. Francis BKN Terriers": "St. Francis (NY)",
    "Mercer Bears": "Mercer",
    "Army Knights": "Army",
    "Lindenwood Lions": "Lindenwood",
    "UAB Blazers": "UAB",
    "North Dakota Fighting Hawks": "North Dakota",
    "VMI Keydets": "VMI",
    "Lamar Cardinals": "Lamar",
    "UC Irvine Anteaters": "UC Irvine",
    "Incarnate Word Cardinals": "Incarnate Word",
    "American Eagles": "American",
    "McNeese Cowboys": "McNeese State",
    "Louisville Cardinals": "Louisville",
    "Tulane Green Wave": "Tulane",
    "Harvard Crimson": "Harvard",
    "Binghamton Bearcats": "Binghamton",
    "Kentucky Wildcats": "Kentucky",
    "Texas Southern Tigers": "Texas Southern",
    "Sacramento St Hornets": "Sacramento State",
    "Northern Kentucky Norse": "Northern Kentucky"
}

In [12]:
## Getting lists of teams in data sets

# Reading in data
odds_df = pd.read_csv("historical_cbb_odds.csv")
stats_df = pd.read_csv("InputData.csv")

odds_teams = set(odds_df["home_team"])
stats_teams= set(stats_df["Team 1"])

with open("odds_teams.txt", "w") as f:
    for team in odds_teams:
        f.write(team + "\n")

with open("stats_teams.txt", "w") as f:
    for team in stats_teams:
        f.write(team + "\n")

In [27]:
## Merging betting data into stats data
from tqdm import tqdm

# Reading in data
odds_df = pd.read_csv("historical_cbb_odds.csv")
stats_df = pd.read_csv("InputData.csv")

odds_df["home_team"] = odds_df["home_team"].map(odds_stats_mapping)
odds_df["away_team"] = odds_df["away_team"].map(odds_stats_mapping)
odds_df["team"] = odds_df["team"].map(odds_stats_mapping)

odds_df = odds_df[odds_df["bookmaker"] == "DraftKings"]

odds_df["snapshot_date"] = pd.to_datetime(odds_df["snapshot_date"], errors="coerce").dt.date
stats_df["Date"] = pd.to_datetime(stats_df["Date"], errors="coerce").dt.date

odds = {}

for index, row in tqdm(odds_df.iterrows(), total=len(odds_df)):
    home_team = row["home_team"]
    away_team = row["away_team"]
    date = row["snapshot_date"]

    stats_row = stats_df[((stats_df["Team 1"] == home_team) & (stats_df["Team 2"] == away_team) & (stats_df["Date"] == date)) |
                   ((stats_df["Team 2"] == home_team) & (stats_df["Team 1"] == away_team) & (stats_df["Date"] == date))]
    
    if not stats_row.empty:
        game_id = stats_row["Game ID"].iloc[0]
        if game_id not in odds:
            odds[game_id] = {row["team"] : row["price"]}
        else:
            odds[game_id][row["team"]] = row["price"]
        

100%|██████████| 45266/45266 [19:08<00:00, 39.41it/s]


In [28]:
def get_team_odds(row, team_column):
    """Looks up the odds for the team specified by team_column using the Game ID."""
    game_id = row["Game ID"]
    team_name = row[team_column]
    
    if game_id in odds and team_name in odds[game_id]:
        return odds[game_id][team_name]
    return pd.NA # Use pd.NA for missing odds

# Create the 'Team 1 Odds' column
stats_df["Team 1 Odds"] = stats_df.apply(
    lambda row: get_team_odds(row, "Team 1"), axis=1
)

# Create the 'Team 2 Odds' column
stats_df["Team 2 Odds"] = stats_df.apply(
    lambda row: get_team_odds(row, "Team 2"), axis=1
)

# Display the first few rows to verify the merge
print(stats_df.head())

# Save the updated DataFrame back to a CSV file
stats_df.to_csv("InputData_with_Odds.csv", index=False)

     Game ID              Team 1              Team 2        Date  Site  \
0  243180167          New Mexico         Santa Clara  2004-11-13  home   
1  243182184            Duquesne  North Carolina A&T  2004-11-13  home   
2  243190167            Duquesne          New Mexico  2004-11-14  away   
3  243192541  North Carolina A&T         Santa Clara  2004-11-14  away   
4  243212717  Jacksonville State    Western Carolina  2004-11-16  away   

   Outcome  is_tournament_game      1-ORtg      1-DRtg    1-3PAr  ...  \
0        1               False  106.837607   83.392226  0.243902  ...   
1        1               False   83.870968   96.858639  0.230769  ...   
2        0               False   93.008542   90.406196  0.269841  ...   
3        0               False   83.920669  104.145937  0.474576  ...   
4        1               False   95.301125  106.241700  0.289855  ...   

     2-ORB%     2-FTr   2-oeFG%   2-oTOV%   2-oDRB%    2-oFTr   2-Pace  \
0  0.245283  0.363636  0.369231  0.154839 